# 02 — EDA sobre anotaciones

Exploración de las anotaciones de la *VitalDB Arrhythmia Database* con foco en la variable objetivo `rhythm_label`.

**Objetivos**

1. Cargar todas las anotaciones disponibles y unirlas con metadata por `case_id`.
2. Distribución global de `rhythm_label` y desbalance entre clases.
3. Conteo por caso: cuántos latidos por `case_id`, cuántos por `rhythm_label`.
4. Distribución de `bad_signal_quality` y solapamiento con `rhythm_label`.
5. Análisis descriptivo **complementario** de `beat_type`.

**Restricción metodológica**

`beat_type` solo se analiza en modo descriptivo. **No se usa como variable predictora en ningún experimento.** Cualquier estadística sobre `beat_type` aquí es exploratoria.

**Filtros base**

- Excluir registros con `bad_signal_quality`.
- Excluir la clase `Noise`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data_loading import load_metadata, load_all_annotations, merge_metadata_and_annotations
from src.preprocessing import apply_basic_filters, exclude_bad_signal_quality, exclude_rhythm_labels

sns.set_theme(context="notebook", style="whitegrid")

## 2. Carga y merge

In [ ]:
metadata = load_metadata()
annotations = load_all_annotations()  # carga todos los archivos disponibles

print("Metadata shape:", metadata.shape)
print("Anotaciones shape:", annotations.shape)

merged = merge_metadata_and_annotations(metadata, annotations, on=config.CASE_ID_COLUMN, how="inner")
print("Merge shape:", merged.shape)
merged.head()

## 3. Aplicación de filtros base

In [ ]:
filtered = apply_basic_filters(
    merged,
    target_column=config.TARGET_COLUMN,
    signal_quality_column=config.SIGNAL_QUALITY_COLUMN,
    excluded_labels=config.EXCLUDED_RHYTHM_LABELS,
)
print("Filas antes:", len(merged))
print("Filas después de filtros base:", len(filtered))

## 4. Distribución global de `rhythm_label`

In [ ]:
rhythm_counts = filtered[config.TARGET_COLUMN].value_counts(dropna=False)
rhythm_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
rhythm_counts.plot(kind="bar", ax=ax)
ax.set_title("Distribución de rhythm_label")
ax.set_xlabel("rhythm_label")
ax.set_ylabel("# latidos anotados")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## 5. Conteos por `case_id`

In [ ]:
beats_per_case = filtered.groupby(config.CASE_ID_COLUMN).size().sort_values(ascending=False)
print("Casos únicos:", beats_per_case.shape[0])
beats_per_case.describe()

In [ ]:
rhythms_per_case = (
    filtered.groupby([config.CASE_ID_COLUMN, config.TARGET_COLUMN]).size().unstack(fill_value=0)
)
rhythms_per_case.head()

## 6. Calidad de señal (`bad_signal_quality`)

Comparar conteos antes y después de aplicar `exclude_bad_signal_quality` sobre los datos no filtrados.

In [ ]:
before = len(merged)
after = len(exclude_bad_signal_quality(merged, column=config.SIGNAL_QUALITY_COLUMN))
print("Antes:", before, "Después de excluir bad_signal_quality:", after)

## 7. `beat_type` (análisis descriptivo complementario)

> Solo descriptivo. No se utiliza como predictor en los modelos.

In [ ]:
if config.BEAT_TYPE_COLUMN in filtered.columns:
    print(filtered[config.BEAT_TYPE_COLUMN].value_counts(dropna=False))
else:
    print(f"La columna '{config.BEAT_TYPE_COLUMN}' no está disponible en los datos cargados.")

## 8. Próximos pasos

- Pasar a `03_ecg_loading_and_visualization.ipynb` para cargar señales crudas desde VitalDB para un subconjunto reducido de `case_id`.